In [1]:
import pandas as pd
import sqlite3
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor, PassiveAggressiveRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

DB_PATH = 'ev_championship.db'

print("=" * 70)
print("МОДУЛЬ В.1: РЕГРЕССИЯ С ПОДДЕРЖКОЙ ИНКРЕМЕНТАЛЬНОГО ОБУЧЕНИЯ")
print("=" * 70)

# ============================================
# 1. ЗАГРУЗКА И АГРЕГАЦИЯ ДАННЫХ
# ============================================

conn = sqlite3.connect(DB_PATH)
telematics = pd.read_sql("""
    SELECT
        vehicle_id,
        timestamp,
        speed_kmh,
        battery_soc_percent,
        consumption_kwh_per_100km,
        acceleration,
        odometer,
        avg_temperature_c,
        road_type
    FROM telematics_preprocessed
    WHERE timestamp IS NOT NULL
      AND consumption_kwh_per_100km BETWEEN 5 AND 50
      AND avg_temperature_c IS NOT NULL
    ORDER BY vehicle_id, timestamp
""", conn)
conn.close()

telematics['timestamp'] = pd.to_datetime(telematics['timestamp'])

# Определение поездок (>30 мин простоя)
telematics = telematics.sort_values(['vehicle_id', 'timestamp'])
telematics['time_diff'] = telematics.groupby('vehicle_id')['timestamp'].diff().dt.total_seconds() / 60
telematics['is_new_trip'] = (telematics['time_diff'] > 30) | (telematics['time_diff'].isna())
telematics['trip_num'] = telematics.groupby('vehicle_id')['is_new_trip'].cumsum()

# Агрегация признаков
trips = telematics.groupby(['vehicle_id', 'trip_num']).agg({
    'speed_kmh': 'mean',
    'battery_soc_percent': ['first', 'last'],
    'consumption_kwh_per_100km': 'mean',
    'acceleration': 'mean',
    'odometer': ['min', 'max'],
    'avg_temperature_c': 'mean',
    'road_type': lambda x: x.mode()[0] if not x.mode().empty else 'unknown'
}).reset_index()

trips.columns = [
    'vehicle_id', 'trip_num',
    'avg_speed_kmh',
    'start_soc', 'end_soc',
    'avg_consumption',
    'avg_acceleration',
    'odometer_start', 'odometer_end',
    'avg_temperature_c',
    'road_type'
]

trips['distance_km'] = (trips['odometer_end'] - trips['odometer_start']).abs()

# ============================================
# 2. ПОДГОТОВКА ПРИЗНАКОВ
# ============================================

# Кодирование типа дороги
road_mapping = {'primary': 0, 'highway': 1, 'rural': 2, 'unknown': 3}
trips['road_type_encoded'] = trips['road_type'].map(road_mapping).fillna(3)

feature_cols = [
    'distance_km',
    'avg_speed_kmh',
    'start_soc',
    'avg_temperature_c',
    'road_type_encoded',
    'avg_acceleration'
]

X = trips[feature_cols].copy()
y = trips['avg_consumption'].copy()

# Очистка от некорректных значений
X = X.replace([np.inf, -np.inf], np.nan)
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

print(f"Подготовлено поездок: {len(X):,}")

# ============================================
# 3. ХОЛД-АУТ ВАЛИДАЦИЯ (80/20)
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nРАЗДЕЛЕНИЕ ДАННЫХ:")
print(f"  • Обучающая выборка (80%): {len(X_train):,} поездок — ТОЛЬКО для обучения")
print(f"  • Холд-аут выборка (20%):  {len(X_test):,} поездок — ТОЛЬКО для тестирования")

# Масштабирование ТОЛЬКО на обучающей выборке
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================================
# 4. ОБУЧЕНИЕ МОДЕЛЕЙ НА ОБУЧАЮЩЕЙ ВЫБОРКЕ
# ============================================

print("\n" + "=" * 70)
print("ОБУЧЕНИЕ МОДЕЛЕЙ НА ОБУЧАЮЩЕЙ ВЫБОРКЕ (80% ДАННЫХ)")
print("=" * 70)

models = {
    'SGDRegressor (squared_error)': SGDRegressor(
        loss='squared_error',
        penalty='l2',
        alpha=0.0001,
        random_state=42,
        max_iter=1000,
        tol=1e-3
    ),
    'SGDRegressor (huber)': SGDRegressor(
        loss='huber',
        epsilon=1.35,
        penalty='l2',
        alpha=0.0001,
        random_state=42,
        max_iter=1000,
        tol=1e-3
    ),
    'PassiveAggressiveRegressor': PassiveAggressiveRegressor(
        C=1.0,
        random_state=42,
        max_iter=1000,
        tol=1e-3
    )
}

results = []

for name, model in models.items():
    # === ЭТАП 1: ОБУЧЕНИЕ ТОЛЬКО НА TRAIN ===
    model.fit(X_train_scaled, y_train)
    print(f"\n{name}")
    print(f"  → Обучение ЗАВЕРШЕНО на {len(X_train)} поездках")

    # === ЭТАП 2: ПРОГНОЗ НА ХОЛД-АУТ ===
    y_pred = model.predict(X_test_scaled)
    print(f"  → Прогноз рассчитан на {len(X_test)} поездках (холд-аут)")

    # === ЭТАП 3: ОЦЕНКА КАЧЕСТВА ТОЛЬКО НА ХОЛД-АУТ ===
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"  → РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ НА ХОЛД-АУТ ВЫБОРКЕ:")
    print(f"      MAE:  {mae:.2f} кВт⋅ч/100км")
    print(f"      RMSE: {rmse:.2f} кВт⋅ч/100км")
    print(f"      R²:   {r2:.4f}")

    results.append({
        'model': name,
        'mae': mae,
        'rmse': rmse,
        'r2': r2
    })

# ============================================
# 5. ВЫБОР ЛУЧШЕЙ МОДЕЛИ ПО РЕЗУЛЬТАТАМ ХОЛД-АУТ
# ============================================

best = min(results, key=lambda x: x['mae'])
best_name = best['model']
print(f"\n" + "=" * 70)
print(f"🏆 ЛУЧШАЯ МОДЕЛЬ ПО РЕЗУЛЬТАТАМ ТЕСТИРОВАНИЯ НА ХОЛД-АУТ:")
print(f"   {best_name}")
print("=" * 70)
print(f"   MAE:  {best['mae']:.2f} кВт⋅ч/100км")
print(f"   RMSE: {best['rmse']:.2f} кВт⋅ч/100км")
print(f"   R²:   {best['r2']:.4f}")
print(f"\n⚠️  ВАЖНО: Все метрики рассчитаны ТОЛЬКО на холд-аут выборке")
print(f"   (20% данных, которые НЕ участвовали в обучении)")

# ============================================
# 6. ФИНАЛЬНОЕ ОБУЧЕНИЕ И СОХРАНЕНИЕ
# ============================================

# Находим лучшую модель
final_model = None
for name, model in models.items():
    if name == best_name:
        final_model = model
        break

# Обучаем на ВСЕХ данных для максимального качества в продакшене
print("\n" + "=" * 70)
print("ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ВСЕХ ДАННЫХ ДЛЯ ПРОДАКШЕНА")
print("=" * 70)
final_model.fit(scaler.transform(X), y)
print(f"✅ Обучение завершено на {len(X):,} поездках")

# Сохранение
model_package = {
    'model': final_model,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'road_mapping': road_mapping,
    'supports_partial_fit': True,
    'metadata': {
        'task': 'incremental_regression',
        'target': 'avg_consumption (кВт⋅ч/100км)',
        'best_model': best_name,
        'mae_holdout': best['mae'],
        'rmse_holdout': best['rmse'],
        'r2_holdout': best['r2'],
        'train_size': len(X_train),
        'holdout_size': len(X_test),
        'total_samples': len(X),
        'features': feature_cols,
        'incremental_learning_ready': True
    }
}

joblib.dump(model_package, 'ev_incremental_regressor_v1.pkl')
print("\n" + "=" * 70)
print("✅ МОДЕЛЬ СОХРАНЕНА: ev_incremental_regressor_v1.pkl")
print("=" * 70)

# ============================================
# 7. ФИНАЛЬНАЯ ДЕМОНСТРАЦИЯ ТЕСТИРОВАНИЯ
# ============================================

print("\n" + "=" * 70)
print("ФИНАЛЬНАЯ ДЕМОНСТРАЦИЯ: ТЕСТИРОВАНИЕ НА ХОЛД-АУТ ВЫБОРКЕ")
print("=" * 70)

# Прогноз лучшей модели на холд-аут
y_pred_final = final_model.predict(scaler.transform(X_test))

# Расчёт метрик
mae_final = mean_absolute_error(y_test, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
r2_final = r2_score(y_test, y_pred_final)

print(f"\nРазмер холд-аут выборки: {len(X_test)} поездок")
print(f"\nМетрики качества на холд-аут выборке:")
print(f"  • MAE (Mean Absolute Error):  {mae_final:.2f} кВт⋅ч/100км")
print(f"  • RMSE (Root Mean Squared Error): {rmse_final:.2f} кВт⋅ч/100км")
print(f"  • R² (Coefficient of Determination): {r2_final:.4f}")

# Примеры прогнозов
print(f"\nПримеры прогнозов (первые 5 поездок из холд-аут):")
print(f"{'Факт':<10} {'Прогноз':<10} {'Ошибка':<10}")
print("-" * 35)
for i in range(min(5, len(y_test))):
    actual = y_test.iloc[i] if isinstance(y_test, pd.Series) else y_test[i]
    pred = y_pred_final[i]
    error = abs(actual - pred)
    print(f"{actual:<10.2f} {pred:<10.2f} {error:<10.2f}")

print("\n" + "=" * 70)
print("✅ МОДУЛЬ В.1 ЗАВЕРШЁН")
print("=" * 70)
print("\nИтоги:")
print(f"  • Рассмотрено моделей: {len(models)}")
print(f"  • Лучшая модель: {best_name}")
print(f"  • MAE на холд-аут: {best['mae']:.2f} кВт⋅ч/100км")
print(f"  • Все модели поддерживают инкрементальное обучение (partial_fit)")
print(f"  • Модель сохранена для модуля В.2")

МОДУЛЬ В.1: РЕГРЕССИЯ С ПОДДЕРЖКОЙ ИНКРЕМЕНТАЛЬНОГО ОБУЧЕНИЯ
Подготовлено поездок: 2,338

РАЗДЕЛЕНИЕ ДАННЫХ:
  • Обучающая выборка (80%): 1,870 поездок — ТОЛЬКО для обучения
  • Холд-аут выборка (20%):  468 поездок — ТОЛЬКО для тестирования

ОБУЧЕНИЕ МОДЕЛЕЙ НА ОБУЧАЮЩЕЙ ВЫБОРКЕ (80% ДАННЫХ)

SGDRegressor (squared_error)
  → Обучение ЗАВЕРШЕНО на 1870 поездках
  → Прогноз рассчитан на 468 поездках (холд-аут)
  → РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ НА ХОЛД-АУТ ВЫБОРКЕ:
      MAE:  7.07 кВт⋅ч/100км
      RMSE: 9.36 кВт⋅ч/100км
      R²:   0.1280

SGDRegressor (huber)
  → Обучение ЗАВЕРШЕНО на 1870 поездках
  → Прогноз рассчитан на 468 поездках (холд-аут)
  → РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ НА ХОЛД-АУТ ВЫБОРКЕ:
      MAE:  7.03 кВт⋅ч/100км
      RMSE: 9.55 кВт⋅ч/100км
      R²:   0.0936

PassiveAggressiveRegressor
  → Обучение ЗАВЕРШЕНО на 1870 поездках
  → Прогноз рассчитан на 468 поездках (холд-аут)
  → РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ НА ХОЛД-АУТ ВЫБОРКЕ:
      MAE:  7.92 кВт⋅ч/100км
      RMSE: 10.43 кВт⋅ч/100км


C:\Users\homje\PycharmProjects\Champion\.venv\Lib\site-packages\sklearn\utils\deprecation.py:71: FutureWarning: Class PassiveAggressiveRegressor is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDRegressor(loss='epsilon_insensitive', penalty=None, learning_rate='pa1', eta0 = 1.0)` instead.
  warnings.warn(msg, category=FutureWarning)


In [2]:
import pandas as pd
import sqlite3
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor, PassiveAggressiveRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from scipy import stats
import joblib
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

DB_PATH = 'ev_championship.db'
MODEL_PATH = 'ev_incremental_regressor_v1.pkl'
LOG_PATH = 'model_update_log.csv'

print("=" * 70)
print("МОДУЛЬ В.2: НЕПРЕРЫВНОЕ ОБУЧЕНИЕ МОДЕЛИ")
print("=" * 70)

# ============================================
# 1. ЗАГРУЗКА ИСТОРИЧЕСКИХ ДАННЫХ И МОДЕЛИ
# ============================================

def load_historical_data():
    """Загрузка исторических данных для сравнения с новыми"""
    conn = sqlite3.connect(DB_PATH)
    telematics = pd.read_sql("""
        SELECT
            vehicle_id,
            timestamp,
            speed_kmh,
            battery_soc_percent,
            consumption_kwh_per_100km,
            acceleration,
            odometer,
            avg_temperature_c,
            road_type
        FROM telematics_preprocessed
        WHERE timestamp IS NOT NULL
          AND consumption_kwh_per_100km BETWEEN 5 AND 50
          AND avg_temperature_c IS NOT NULL
        ORDER BY vehicle_id, timestamp
    """, conn)
    conn.close()

    telematics['timestamp'] = pd.to_datetime(telematics['timestamp'])
    telematics = telematics.sort_values(['vehicle_id', 'timestamp'])
    telematics['time_diff'] = telematics.groupby('vehicle_id')['timestamp'].diff().dt.total_seconds() / 60
    telematics['is_new_trip'] = (telematics['time_diff'] > 30) | (telematics['time_diff'].isna())
    telematics['trip_num'] = telematics.groupby('vehicle_id')['is_new_trip'].cumsum()

    trips = telematics.groupby(['vehicle_id', 'trip_num']).agg({
        'speed_kmh': 'mean',
        'battery_soc_percent': ['first', 'last'],
        'consumption_kwh_per_100km': 'mean',
        'acceleration': 'mean',
        'odometer': ['min', 'max'],
        'avg_temperature_c': 'mean',
        'road_type': lambda x: x.mode()[0] if not x.mode().empty else 'unknown'
    }).reset_index()

    trips.columns = [
        'vehicle_id', 'trip_num',
        'avg_speed_kmh',
        'start_soc', 'end_soc',
        'avg_consumption',
        'avg_acceleration',
        'odometer_start', 'odometer_end',
        'avg_temperature_c',
        'road_type'
    ]

    trips['distance_km'] = (trips['odometer_end'] - trips['odometer_start']).abs()
    road_mapping = {'primary': 0, 'highway': 1, 'rural': 2, 'unknown': 3}
    trips['road_type_encoded'] = trips['road_type'].map(road_mapping).fillna(3)

    feature_cols = [
        'distance_km',
        'avg_speed_kmh',
        'start_soc',
        'avg_temperature_c',
        'road_type_encoded',
        'avg_acceleration'
    ]

    X = trips[feature_cols].copy()
    y = trips['avg_consumption'].copy()

    X = X.replace([np.inf, -np.inf], np.nan)
    mask = X.notna().all(axis=1) & y.notna()
    X = X[mask]
    y = y[mask]

    return X, y, feature_cols, road_mapping

def load_model(path):
    """Загрузка сохранённой модели из В.1"""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Модель не найдена: {path}. Сначала выполните модуль В.1")

    package = joblib.load(path)
    return package['model'], package['scaler'], package['feature_cols'], package['metadata']

# Загрузка исторических данных и модели
print("Загрузка исторических данных...")
X_hist, y_hist, feature_cols, road_mapping = load_historical_data()
print(f"  Исторических поездок: {len(X_hist):,}")

print("\nЗагрузка модели из В.1...")
try:
    model, scaler, model_features, metadata = load_model(MODEL_PATH)
    print(f"  Модель: {metadata['best_model']}")
    print(f"  Поддержка инкрементального обучения: {hasattr(model, 'partial_fit')}")
except FileNotFoundError as e:
    print(f"❌ ОШИБКА: {e}")
    print("Выполните сначала модуль В.1 для создания модели.")
    exit(1)

# ============================================
# 2. СИМУЛЯЦИЯ ПОСТУПЛЕНИЯ НОВЫХ ДАННЫХ
# ============================================

print("\n" + "=" * 70)
print("СИМУЛЯЦИЯ ПОСТУПЛЕНИЯ НОВЫХ ДАННЫХ")
print("=" * 70)

# Разделение исторических данных: 80% — "старые", 20% — "новые" (симуляция)
X_old, X_new_sim, y_old, y_new_sim = train_test_split(
    X_hist, y_hist, test_size=0.2, random_state=42
)

print(f"  Старые данные (обучение В.1): {len(X_old):,} поездок")
print(f"  Новые данные (поступили сейчас): {len(X_new_sim):,} поездок")

# Инжекция дрейфа для демонстрации (повышение температуры на 15°C)
print("\nИнжекция дрейфа в новые данные (повышение температуры на 15°C)...")
X_new_sim_drift = X_new_sim.copy()
X_new_sim_drift['avg_temperature_c'] += 15.0  # искусственный дрейф

# ============================================
# 3. МОНИТОРИНГ ДРЕЙФА ДАННЫХ
# ============================================

def detect_feature_drift(old_data, new_data, feature, threshold=0.05):
    """
    Обнаружение дрейфа для одного признака через тест Колмогорова-Смирнова
    Возвращает: (дрейф_обнаружен, p_value, статистика)
    """
    # Убираем бесконечности и пропуски
    old_clean = old_data[feature].replace([np.inf, -np.inf], np.nan).dropna()
    new_clean = new_data[feature].replace([np.inf, -np.inf], np.nan).dropna()

    if len(old_clean) < 10 or len(new_clean) < 10:
        return False, 1.0, 0.0

    stat, p_value = stats.ks_2samp(old_clean, new_clean)
    drift_detected = p_value < threshold

    return drift_detected, p_value, stat

def monitor_drift(old_data, new_data, features, threshold=0.05):
    """
    Мониторинг дрейфа по всем ключевым признакам
    Возвращает: (общий_дрейф, детали_по_признакам)
    """
    drift_details = []
    overall_drift = False

    for feature in features:
        drift, p_val, stat = detect_feature_drift(old_data, new_data, feature, threshold)
        drift_details.append({
            'feature': feature,
            'drift_detected': drift,
            'p_value': p_val,
            'statistic': stat
        })
        if drift:
            overall_drift = True

    return overall_drift, drift_details

print("\n" + "=" * 70)
print("МОНИТОРИНГ ДРЕЙФА ДАННЫХ")
print("=" * 70)

# Ключевые признаки для мониторинга
monitor_features = ['avg_temperature_c', 'avg_speed_kmh', 'distance_km']

# Проверка дрейфа БЕЗ инжекции (базовый сценарий)
print("\n[БАЗОВЫЙ СЦЕНАРИЙ] Проверка дрейфа без искусственных изменений:")
drift_base, details_base = monitor_drift(X_old, X_new_sim, monitor_features, threshold=0.05)

for detail in details_base:
    status = "⚠️ ДРЕЙФ" if detail['drift_detected'] else "✅ Норма"
    print(f"  • {detail['feature']:25s} | {status} | p-value={detail['p_value']:.4f} | статистика={detail['statistic']:.4f}")

print(f"\nИтог базового сценария: {'⚠️ ДРЕЙФ ОБНАРУЖЕН' if drift_base else '✅ Дрейф не обнаружен'}")

# Проверка дрейфа С инжекцией (для демонстрации)
print("\n[СЦЕНАРИЙ С ДРЕЙФОМ] Проверка дрейфа с искусственным повышением температуры:")
drift_injected, details_injected = monitor_drift(X_old, X_new_sim_drift, monitor_features, threshold=0.05)

for detail in details_injected:
    status = "⚠️ ДРЕЙФ" if detail['drift_detected'] else "✅ Норма"
    print(f"  • {detail['feature']:25s} | {status} | p-value={detail['p_value']:.4f} | статистика={detail['statistic']:.4f}")

print(f"\nИтог сценария с дрейфом: {'⚠️ ДРЕЙФ ОБНАРУЖЕН' if drift_injected else '✅ Дрейф не обнаружен'}")

# ============================================
# 4. МЕХАНИЗМ ОБНОВЛЕНИЯ МОДЕЛИ
# ============================================

def update_model(new_X, new_y, drift_detected, model, scaler, feature_cols, road_mapping, metadata):
    """
    Обновление модели в зависимости от наличия дрейфа:
    - При дрейфе: полное переобучение на объединённых данных
    - Без дрейфа: инкрементальное дообучение через partial_fit
    """
    update_type = "full_retrain" if drift_detected else "incremental"

    if drift_detected:
        print("\n⚠️  ДРЕЙФ ОБНАРУЖЕН → ПОЛНОЕ ПЕРЕОБУЧЕНИЕ МОДЕЛИ")
        # Объединение старых и новых данных
        X_combined = pd.concat([X_old, new_X], ignore_index=True)
        y_combined = pd.concat([y_old, new_y], ignore_index=True)

        # Масштабирование
        X_scaled = scaler.fit_transform(X_combined)

        # Создание новой модели того же типа
        if isinstance(model, SGDRegressor):
            new_model = SGDRegressor(
                loss=model.loss,
                penalty=model.penalty,
                alpha=model.alpha,
                random_state=42,
                max_iter=1000,
                tol=1e-3
            )
        elif isinstance(model, PassiveAggressiveRegressor):
            new_model = PassiveAggressiveRegressor(
                C=model.C,
                random_state=42,
                max_iter=1000,
                tol=1e-3
            )
        else:
            raise ValueError("Неизвестный тип модели для переобучения")

        # Полное обучение
        new_model.fit(X_scaled, y_combined)

    else:
        print("\n✅ ДРЕЙФ НЕ ОБНАРУЖЕН → ИНКРЕМЕНТАЛЬНОЕ ДООБУЧЕНИЕ")
        # Масштабирование новых данных
        X_new_scaled = scaler.transform(new_X)

        # Инкрементальное обучение
        model.partial_fit(X_new_scaled, new_y)
        new_model = model

    # Оценка качества на холд-аут (симуляция)
    X_eval, X_holdout, y_eval, y_holdout = train_test_split(
        new_X, new_y, test_size=0.3, random_state=42
    )
    X_holdout_scaled = scaler.transform(X_holdout)
    y_pred = new_model.predict(X_holdout_scaled)

    mae = mean_absolute_error(y_holdout, y_pred)
    rmse = np.sqrt(mean_squared_error(y_holdout, y_pred))
    r2 = r2_score(y_holdout, y_pred)

    # Определение версии
    version = 1
    while os.path.exists(f"ev_model_v{version}.pkl"):
        version += 1

    # Сохранение модели
    model_package = {
        'model': new_model,
        'scaler': scaler,
        'feature_cols': feature_cols,
        'road_mapping': road_mapping,
        'supports_partial_fit': hasattr(new_model, 'partial_fit'),
        'metadata': {
            'task': 'incremental_regression',
            'target': 'avg_consumption (кВт⋅ч/100км)',
            'best_model': type(new_model).__name__,
            'mae_holdout': mae,
            'rmse_holdout': rmse,
            'r2_holdout': r2,
            'update_type': update_type,
            'drift_detected': drift_detected,
            'new_samples': len(new_X),
            'total_samples': len(X_old) + len(new_X),
            'update_timestamp': datetime.now().isoformat(),
            'version': version
        }
    }

    model_path = f"ev_model_v{version}.pkl"
    joblib.dump(model_package, model_path)

    # Логирование
    log_entry = {
        'version': version,
        'timestamp': datetime.now().isoformat(),
        'update_type': update_type,
        'drift_detected': drift_detected,
        'new_samples': len(new_X),
        'total_samples': len(X_old) + len(new_X),
        'mae': mae,
        'rmse': rmse,
        'r2': r2
    }

    # Сохранение лога
    if os.path.exists(LOG_PATH):
        log_df = pd.read_csv(LOG_PATH)
        log_df = pd.concat([log_df, pd.DataFrame([log_entry])], ignore_index=True)
    else:
        log_df = pd.DataFrame([log_entry])

    log_df.to_csv(LOG_PATH, index=False)

    return version, mae, rmse, r2, update_type, model_path

# ============================================
# 5. ДЕМОНСТРАЦИЯ ОБНОВЛЕНИЯ МОДЕЛИ
# ============================================

print("\n" + "=" * 70)
print("ДЕМОНСТРАЦИЯ ОБНОВЛЕНИЯ МОДЕЛИ")
print("=" * 70)

# Сценарий 1: Без дрейфа (инкрементальное обучение)
print("\n[СЦЕНАРИЙ 1] Обновление без дрейфа (инкрементальное обучение):")
version1, mae1, rmse1, r21, type1, path1 = update_model(
    X_new_sim, y_new_sim, drift_detected=False,
    model=model, scaler=scaler, feature_cols=feature_cols,
    road_mapping=road_mapping, metadata=metadata
)

print(f"  ✅ Версия сохранена: {path1}")
print(f"  • Тип обновления: {type1}")
print(f"  • MAE:  {mae1:.2f} кВт⋅ч/100км")
print(f"  • RMSE: {rmse1:.2f} кВт⋅ч/100км")
print(f"  • R²:   {r21:.4f}")

# Загрузка обновлённой модели для следующего сценария
model_v1, scaler_v1, _, _ = load_model(path1)

# Сценарий 2: С дрейфом (полное переобучение)
print("\n[СЦЕНАРИЙ 2] Обновление с дрейфом (полное переобучение):")
version2, mae2, rmse2, r22, type2, path2 = update_model(
    X_new_sim_drift, y_new_sim, drift_detected=True,
    model=model_v1, scaler=scaler_v1, feature_cols=feature_cols,
    road_mapping=road_mapping, metadata=metadata
)

print(f"  ✅ Версия сохранена: {path2}")
print(f"  • Тип обновления: {type2}")
print(f"  • MAE:  {mae2:.2f} кВт⋅ч/100км")
print(f"  • RMSE: {rmse2:.2f} кВт⋅ч/100км")
print(f"  • R²:   {r22:.4f}")

# ============================================
# 6. ИТОГОВЫЙ ОТЧЁТ
# ============================================

print("\n" + "=" * 70)
print("ИТОГОВЫЙ ОТЧЁТ ПО МОДУЛЮ В.2")
print("=" * 70)

log_df = pd.read_csv(LOG_PATH)
print(f"\nВерсии моделей:")
for _, row in log_df.iterrows():
    drift_str = "⚠️ ДРЕЙФ" if row['drift_detected'] else "✅ Норма"
    print(f"  • v{row['version']:2d} | {row['update_type']:15s} | {drift_str} | "
          f"MAE={row['mae']:.2f} | образцов={row['total_samples']} | {row['timestamp'][:16]}")

print("\n✅ МОДУЛЬ В.2 ЗАВЕРШЁН")
print("\nРеализовано:")
print("  • Механизм дообучения модели при поступлении новых данных (partial_fit)")
print("  • Мониторинг дрейфа данных через тест Колмогорова-Смирнова")
print("  • Автоматическое полное переобучение при обнаружении дрейфа")
print("  • Версионирование моделей (ev_model_v1.pkl, ev_model_v2.pkl, ...)")
print("  • Логирование всех обновлений в model_update_log.csv")
print("\nЛог обновлений сохранён: model_update_log.csv")

МОДУЛЬ В.2: НЕПРЕРЫВНОЕ ОБУЧЕНИЕ МОДЕЛИ
Загрузка исторических данных...
  Исторических поездок: 2,338

Загрузка модели из В.1...
  Модель: SGDRegressor (huber)
  Поддержка инкрементального обучения: True

СИМУЛЯЦИЯ ПОСТУПЛЕНИЯ НОВЫХ ДАННЫХ
  Старые данные (обучение В.1): 1,870 поездок
  Новые данные (поступили сейчас): 468 поездок

Инжекция дрейфа в новые данные (повышение температуры на 15°C)...

МОНИТОРИНГ ДРЕЙФА ДАННЫХ

[БАЗОВЫЙ СЦЕНАРИЙ] Проверка дрейфа без искусственных изменений:
  • avg_temperature_c         | ✅ Норма | p-value=0.6418 | статистика=0.0378
  • avg_speed_kmh             | ✅ Норма | p-value=0.0807 | статистика=0.0650
  • distance_km               | ✅ Норма | p-value=0.7065 | статистика=0.0358

Итог базового сценария: ✅ Дрейф не обнаружен

[СЦЕНАРИЙ С ДРЕЙФОМ] Проверка дрейфа с искусственным повышением температуры:
  • avg_temperature_c         | ⚠️ ДРЕЙФ | p-value=0.0000 | статистика=0.5016
  • avg_speed_kmh             | ✅ Норма | p-value=0.0807 | статистика=0.0650

In [4]:
import pandas as pd
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import os

DB_PATH = 'ev_championship.db'
OUTPUT_DIR = 'battery_forecasts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("МОДУЛЬ В.3: ПРОГНОЗИРОВАНИЕ ДЕГРАДАЦИИ БАТАРЕИ")
print("=" * 70)

# ============================================
# 1. ЗАГРУЗКА ДАННЫХ ТОЛЬКО ИЗ vehicles
# ============================================

conn = sqlite3.connect(DB_PATH)
vehicles = pd.read_sql("""
    SELECT
        vehicle_id,
        model,
        battery_capacity_kwh,
        year_of_manufacture,
        battery_health,          -- состояние ПРИ ПОСТУПЛЕНИИ (0.773 = 77.3%)
        initial_odometer_km,     -- пробег ПРИ ПОСТУПЛЕНИИ (60235 км)
        has_problem
    FROM vehicles
    ORDER BY vehicle_id
""", conn)
conn.close()

print(f"Обработано автомобилей: {len(vehicles)}\n")

# ============================================
# 2. РАСЧЁТ ДЕГРАДАЦИИ И ПРОГНОЗА (ТОЛЬКО ИЗ vehicles)
# ============================================

results = []

for idx, row in vehicles.iterrows():
    vid = row['vehicle_id']
    model = row['model']

    # Данные из таблицы vehicles
    health_at_entry = row['battery_health'] * 100  # 77.3%
    odometer_at_entry = row['initial_odometer_km']  # 60235 км

    # === РАСЧЁТ СКОРОСТИ ДЕГРАДАЦИИ ===
    # От 0 км (100%) до момента поступления в парк:
    degradation_to_entry = 100.0 - health_at_entry  # 100% - 77.3% = 22.7%
    km_to_entry = odometer_at_entry                 # 60235 км
    degradation_rate = (degradation_to_entry / km_to_entry) * 1000  # % на 1000 км

    # === РАСЧЁТ ОСТАТОЧНОГО РЕСУРСА ДО 80% ===
    # От текущего состояния (при поступлении) до 80%:
    remaining_to_80 = health_at_entry - 80.0  # сколько % осталось до порога
    if remaining_to_80 > 0:
        remaining_km = (remaining_to_80 / degradation_rate) * 1000
        remaining_years = remaining_km / 15000  # при 15 тыс. км/год
    else:
        remaining_km = 0
        remaining_years = 0

    # === ПРОГНОЗ НА 1 ГОД ===
    # Деградация за 15 000 км:
    degradation_1y = degradation_rate * 15  # % за год
    health_1y = health_at_entry - degradation_1y

    # Сохранение результатов
    results.append({
        'vehicle_id': vid,
        'model': model,
        'odometer_at_entry_km': odometer_at_entry,
        'health_at_entry_pct': health_at_entry,
        'degradation_rate_per_1000km': degradation_rate,
        'remaining_km_to_80': remaining_km,
        'remaining_years_to_80': remaining_years,
        'forecast_health_1y': health_1y,
        'has_problem': row['has_problem']
    })

    # === ВИЗУАЛИЗАЦИЯ ===
    # Точки для графика: 0 км → поступление в парк → прогноз на 1 год → точка 80%
    x = [0, odometer_at_entry]
    y = [100.0, health_at_entry]

    # Прогнозная линия до 80% или до 200 000 км
    if remaining_km > 0:
        x_forecast = np.linspace(odometer_at_entry, odometer_at_entry + remaining_km, 50)
        y_forecast = health_at_entry - (x_forecast - odometer_at_entry) / 1000 * degradation_rate
    else:
        x_forecast = np.linspace(odometer_at_entry, odometer_at_entry + 50000, 50)
        y_forecast = health_at_entry - (x_forecast - odometer_at_entry) / 1000 * degradation_rate

    plt.figure(figsize=(10, 6))

    # История: 0 км → поступление
    plt.plot(x, y, 'g-o', linewidth=2.5, markersize=10, label='История: 0 км → поступление в парк', zorder=3)

    # Прогноз
    plt.plot(x_forecast, y_forecast, 'b--', linewidth=2, label='Прогноз деградации', zorder=2)

    # Линия 80%
    plt.axhline(y=80, color='red', linestyle='--', linewidth=2, label='Порог 80% (конец ресурса)', zorder=1)

    # Точка поступления
    plt.scatter([odometer_at_entry], [health_at_entry], c='green', s=150,
               marker='o', zorder=5, label=f'Поступление: {health_at_entry:.1f}% @ {odometer_at_entry:,.0f} км')

    # Прогноз на 1 год
    odometer_1y = odometer_at_entry + 15000
    health_1y_plot = health_at_entry - (15000 / 1000 * degradation_rate)
    plt.scatter([odometer_1y], [health_1y_plot], c='purple', s=180, marker='*', zorder=5,
               label=f'Прогноз на 1 год: {health_1y_plot:.1f}%')

    # Точка 80%
    if remaining_km > 0:
        odometer_80 = odometer_at_entry + remaining_km
        plt.scatter([odometer_80], [80], c='red', s=200, marker='X', zorder=5,
                   label=f'80% на {odometer_80:,.0f} км ({remaining_years:.1f} лет)')

    plt.xlabel('Пробег (км)', fontsize=12)
    plt.ylabel('Здоровье батареи (%)', fontsize=12)
    plt.title(f'{vid} ({model})\nДеградация: {degradation_rate:.3f}% на 1000 км | До 80%: {remaining_km:,.0f} км ({remaining_years:.1f} лет)',
             fontsize=13, fontweight='bold')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.xlim(left=-10000)
    plt.ylim(bottom=0, top=105)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/battery_degradation_{vid}.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Вывод в консоль
    status = "⚠️ <80%" if health_at_entry < 80 else ("🟡 <90%" if health_at_entry < 90 else "✅ >90%")
    print(f"{vid:6s} | {model:25s} | "
          f"Деградация: {degradation_rate:.3f}%/1000км | "
          f"Поступление: {health_at_entry:.1f}% @ {odometer_at_entry:,.0f} км | "
          f"До 80%: {remaining_km:,.0f} км ({remaining_years:.1f} лет) | "
          f"Через год: {health_1y:.1f}% | {status}")

# ============================================
# 3. ИТОГОВЫЙ ОТЧЁТ
# ============================================

report = pd.DataFrame(results)
report = report.sort_values('remaining_km_to_80')

print("\n" + "=" * 70)
print("ТОП-5 АВТОМОБИЛЕЙ С НАИМЕНЬШИМ РЕСУРСОМ (ДО 80%)")
print("=" * 70)
for _, r in report.head(5).iterrows():
    status = "⚠️ КРИТИЧЕСКИЙ" if r['remaining_km_to_80'] < 30000 else "🟡 ВНИМАНИЕ"
    print(f"{r['vehicle_id']:6s} | {r['model']:25s} | "
          f"До 80%: {r['remaining_km_to_80']:>8,.0f} км ({r['remaining_years_to_80']:>4.1f} лет) | "
          f"Поступление: {r['health_at_entry_pct']:>5.1f}% | {status}")

print("\n" + "=" * 70)
print("ТОП-5 АВТОМОБИЛЕЙ С НАИБОЛЬШИМ РЕСУРСОМ")
print("=" * 70)
for _, r in report.tail(5).iterrows():
    print(f"{r['vehicle_id']:6s} | {r['model']:25s} | "
          f"До 80%: {r['remaining_km_to_80']:>8,.0f} км ({r['remaining_years_to_80']:>4.1f} лет) | "
          f"Поступление: {r['health_at_entry_pct']:>5.1f}%")

# Сохранение отчёта
report.to_csv(f'{OUTPUT_DIR}/battery_forecast_report.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ Отчёт сохранён: {OUTPUT_DIR}/battery_forecast_report.csv")
print(f"✅ Графики сохранены: {OUTPUT_DIR}/battery_degradation_*.png")

МОДУЛЬ В.3: ПРОГНОЗИРОВАНИЕ ДЕГРАДАЦИИ БАТАРЕИ
Обработано автомобилей: 26

EV001  | Skoda Enyaq iV 80         | Деградация: 0.377%/1000км | Поступление: 77.3% @ 60,235 км | До 80%: 0 км (0.0 лет) | Через год: 71.6% | ⚠️ <80%
EV002  | Mercedes EQC 400          | Деградация: 1.355%/1000км | Поступление: 71.0% @ 21,398 км | До 80%: 0 км (0.0 лет) | Через год: 50.7% | ⚠️ <80%
EV003  | Tesla Model 3 LR          | Деградация: 0.356%/1000км | Поступление: 76.6% @ 65,707 км | До 80%: 0 км (0.0 лет) | Через год: 71.3% | ⚠️ <80%
EV004  | Renault Zoe R135          | Деградация: 0.103%/1000км | Поступление: 98.7% @ 12,666 км | До 80%: 182,196 км (12.1 лет) | Через год: 97.2% | ✅ >90%
EV005  | Renault Zoe R135          | Деградация: 0.128%/1000км | Поступление: 94.2% @ 45,217 км | До 80%: 110,704 км (7.4 лет) | Через год: 92.3% | ✅ >90%
EV006  | BMW i3 120Ah              | Деградация: 0.106%/1000км | Поступление: 94.9% @ 48,301 км | До 80%: 141,115 км (9.4 лет) | Через год: 93.3% | ✅ >90%
EV007  | 